In [ ]:
import wandb
wandb.login()

# Download an artifact from wandb
api = wandb.Api()
artifact = api.artifact("coactivelearning/llama_3_8b_ultrafeedback/kl_5e-3_rloo_patch:v0")

# Download the files to a local directory
artifact_dir = "./ultrafeedback_rloo_patch"
artifact.download("./ultrafeedback_rloo_patch")

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: wz-ml (coactivelearning). Use `wandb login --relogin` to force relogin
wandb: Downloading large artifact kl_5e-3_rloo_patch:v0, 432.52MB. 6 files... 
wandb:   6 of 6 files downloaded.  
Done. 0:0:14.7


In [4]:
import torch
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    GenerationConfig,
    PretrainedConfig,
    PreTrainedModel,
    get_scheduler,
    AutoTokenizer
)
from peft import get_peft_model, LoraConfig, PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

config = LoraConfig.from_pretrained(artifact_dir)
policy = PeftModel.from_pretrained(base_model, artifact_dir)

policy.generation_config.eos_token_id = None  # disable `pad_token_id` and `eos_token_id` because we want to generate tokens without truncation / padding
policy.generation_config.pad_token_id = None 

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.20it/s]


In [5]:
merged_model = policy.merge_and_unload()
merged_model.save_pretrained("ultrafeedback_rloo_patch_merged", safe_serialization=False)
del policy; del merged_model; del base_model

[2025-03-31 16:15:40,822] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/will/.conda/envs/dips/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
No ROCm runtime is found, using ROCM_HOME='/opt/rocm'
/home/will/.conda/envs/dips/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/will/.conda/envs/dips/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/will/.conda/envs/dips/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/will/.conda/envs/dips/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/will/.conda/envs/dips/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/wi

KeyboardInterrupt: 

In [5]:
!cp ultrafeedback_rloo_patch/tokenizer.json ultrafeedback_rloo_patch_merged/tokenizer.json
!cp ultrafeedback_rloo_patch/tokenizer_config.json ultrafeedback_rloo_patch_merged/tokenizer_config.json

In [2]:
from vllm import LLM, SamplingParams
query_length = 256
response_length = 1024
template_length = 64
llm = LLM(model="merged_model", 
          task="generate", 
          max_model_len = query_length + response_length + template_length + 1, 
          tensor_parallel_size = 4, 
          gpu_memory_utilization = 0.8)

/home/will/.conda/envs/dips/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 03-27 22:28:45 [__init__.py:239] Automatically detected platform cuda.


2025-03-27 22:28:46,157	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 03-27 22:28:55 [config.py:1519] Defaulting to use mp for distributed inference
INFO 03-27 22:28:55 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-27 22:28:57 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='merged_model', speculative_config=None, tokenizer='merged_model', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=1345, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=4, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar', reasoning_backend=None), observability_config=ObservabilityConfig(show_hidden_metrics=False, otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=None,

Loading pt checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading pt checkpoint shards:  25% Completed | 1/4 [00:04<00:13,  4.61s/it]
Loading pt checkpoint shards:  50% Completed | 2/4 [00:08<00:08,  4.17s/it]
Loading pt checkpoint shards:  75% Completed | 3/4 [00:12<00:04,  4.09s/it]
Loading pt checkpoint shards: 100% Completed | 4/4 [00:13<00:00,  2.79s/it]
Loading pt checkpoint shards: 100% Completed | 4/4 [00:13<00:00,  3.32s/it]
(VllmWorker rank=0 pid=441430) 


(VllmWorker rank=0 pid=441430) INFO 03-27 22:29:16 [loader.py:447] Loading weights took 13.42 seconds
(VllmWorker rank=3 pid=441560) INFO 03-27 22:29:16 [loader.py:447] Loading weights took 13.47 seconds
(VllmWorker rank=1 pid=441474) INFO 03-27 22:29:16 [loader.py:447] Loading weights took 13.61 seconds
(VllmWorker rank=0 pid=441430) INFO 03-27 22:29:16 [gpu_model_runner.py:1186] Model loading took 3.7711 GB and 13.722032 seconds
(VllmWorker rank=3 pid=441560) INFO 03-27 22:29:16 [gpu_model_runner.py:1186] Model loading took 3.7711 GB and 13.768611 seconds
(VllmWorker rank=2 pid=441497) INFO 03-27 22:29:16 [loader.py:447] Loading weights took 13.78 seconds
(VllmWorker rank=1 pid=441474) INFO 03-27 22:29:16 [gpu_model_runner.py:1186] Model loading took 3.7711 GB and 13.919341 seconds
(VllmWorker rank=2 pid=441497) INFO 03-27 22:29:16 [gpu_model_runner.py:1186] Model loading took 3.7711 GB and 14.085754 seconds
(VllmWorker rank=3 pid=441560) (VllmWorker rank=0 pid=441430) (VllmWorker ra

In [3]:
from datasets import load_dataset
ds = load_dataset("openbmb/UltraFeedback")

In [36]:
from transformers import AutoTokenizer
from tqdm import tqdm, trange

ultrafeedback_instructions = ds["train"]["instruction"]
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
tokenizer.padding_side = "left"
pairs = [(tokenizer.apply_chat_template([{"role": "user", "content": instruction}],
                                                 add_generation_prompt=True,
                                                  padding = "max_length",
                                                  max_length = query_length + template_length,
                                                  truncation = True), instruction) for instruction in tqdm(ultrafeedback_instructions)
                                                  if len(tokenizer(instruction).input_ids) <= query_length]

prompt_token_ids, instructions = zip(*pairs)
print(f"Sampling {len(prompt_token_ids)} instructions.")

100%|██████████| 63967/63967 [00:40<00:00, 1576.21it/s]

Sampling 52639 instructions.


In [24]:
sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens = response_length)
outputs = llm.generate(prompt_token_ids=prompt_token_ids[:20], sampling_params=sampling_params)

Processed prompts: 100%|██████████| 20/20 [00:17<00:00,  1.13it/s, est. speed input: 362.20 toks/s, output: 1024.84 toks/s]


In [38]:
import json
alpaca_eval_outputs = []
for index in range(len(outputs)):
    output = {"instruction": instructions[index], "output": outputs[index].outputs[0].text}
    alpaca_eval_outputs.append(output)

with open("alpaca_eval_outputs.json", "w") as f:
    json.dump(alpaca_eval_outputs, f)

In [1]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

/home/will/.conda/envs/dips/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tok.add_special_tokens({"pad_token": "[PAD]"})

1

In [3]:
tok.pad_token_id

128256

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("RLHFlow/ArmoRM-Llama3-8B-v0.1", torch_dtype="auto", device_map="auto")
num_embeddings = model.get_input_embeddings().num_embeddings

Loading checkpoint shards: 100%|██████████| 4/4 [01:46<00:00, 26.74s/it]


In [6]:
num_embeddings

128256